# Research 05 — CatBoost one-vs-rest objective

**Hypothesis:** `MultiClassOneVsAll` may improve macro F1 by learning each segment as a binary distinction. One fixed architecture, same deterministic three-fold OOF split, no parameter search, no Test access. The CatBoost run must pass categorical feature names explicitly.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
base = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(base) == len(train) and np.array_equal(base.segment.to_numpy(), y)
p_base = base[[f"prob_{i}" for i in range(4)]].to_numpy()
X = train.drop(columns=["segment", "player_id"]).copy()
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
X[cat_cols] = X[cat_cols].fillna("__MISSING__").astype(str)
print(f"Training rows={len(train):,}; categorical features={len(cat_cols)}")

Training rows=104,000; categorical features=10


In [2]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
p_ova = np.zeros((len(train), 4), dtype=float)
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    model = CatBoostClassifier(
        iterations=300,
        depth=7,
        learning_rate=0.08,
        loss_function="MultiClassOneVsAll",
        random_seed=42,
        thread_count=2,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(X.iloc[fit_idx], y[fit_idx], cat_features=cat_cols)
    p_ova[valid_idx] = model.predict_proba(X.iloc[valid_idx])
    print(
        f"Fold {fold}: macro F1={f1_score(y[valid_idx], p_ova[valid_idx].argmax(1), average='macro'):.5f}",
        flush=True,
    )

p_blend = 0.8 * p_base + 0.2 * p_ova


def score(name, probabilities):
    pred = probabilities.argmax(axis=1)
    return {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }


results = pd.DataFrame(
    [
        score("Current CatBoost/LightGBM", p_base),
        score("CatBoost OneVsAll", p_ova),
        score("80% current + 20% OneVsAll", p_blend),
    ]
)
display(results)
results.to_csv(ROOT / "assets/segment-catboost-ova-results.csv", index=False)
pd.DataFrame(p_ova, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-catboost-ova-oof.csv", index=False
)
pd.DataFrame(p_blend, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-catboost-ova-blend-oof.csv", index=False
)

Fold 1: macro F1=0.74236


Fold 2: macro F1=0.74278


Fold 3: macro F1=0.74644


,candidate,macro_f1,weighted_f1
0,Current CatBoost/LightGBM,0.750895,0.764683
1,CatBoost OneVsAll,0.743872,0.758647
2,80% current + 20% OneVsAll,0.750310,0.764226


## Decision

Compare OOF results to the current blend (macro F1 0.750895, weighted 0.764683). Treat a small change as inconclusive due repeated experimentation on the same folds; do not promote a candidate solely on this screen.